# MNIST: открытая Wasserstein/tube-дистилляция после проверки лучей

Unconditional pixel FM: **t=1 — шум, t=0 — данные**, `x_next = x - H*v(x,t)`.
Показываем M0/M1/E0/E1 для 32→16: два teacher microsteps против одного student step.
Метки, CFG, VAE и conditioning на H не используются. Ученики — точные копии одного
зафиксированного U-Net teacher, а не случайные новые модели.

**По умолчанию `RUN_TRAINING=False`: долгие MNIST-запуски не стартуют автоматически.**
Сначала должны пройти вращающиеся лучи и их анимации. Ниже показаны точные CLI-команды
для evaluator/teacher и основной зарегистрированной серии. После появления обученного
teacher можно включить короткую **exploratory** дистилляцию прямо здесь: targets, tube,
full-image features, loss, backward, optimizer и EMA физически видны в ячейках.
Это 100 updates на один paired seed; это не подмена зарегистрированных 12 обучений.

Источники и ограничения: [протокол проекта](../replication/README.md),
[исходное ТЗ](../replication/source/REPLICATION_SPEC.md). Веса и checkpoints лежат отдельными
файлами, в notebook нет embedded wheel/base64 payload. До MNIST используйте
[notebook 10 с лучами](10_wasserstein_tube_rays.ipynb).

In [ ]:
import os
import subprocess
import sys

GIT_BRANCH = "research/wasserstein-tube-replication"
GIT_SOURCE = (
    "diffusion-lab[replication,mnist,interactive] @ "
    "git+https://github.com/Vovanm88/ddpm_vibe_research.git@" + GIT_BRANCH
)
if os.environ.get("DIFFUSION_LAB_SKIP_INSTALL") != "1":
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", GIT_SOURCE])


In [ ]:
import copy
import hashlib
import json
import math
import shlex
import tempfile
import time
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import Image, Markdown, clear_output, display
from matplotlib import animation

from recheck.config import load_config
from recheck.data import evaluator_preprocess
from recheck.distill import calibrate, experiment_schedule
from recheck.fm import benchmark_data, learning_rate, load_model, sample_euler
from recheck.losses import production_sinkhorn, sliced_wasserstein_loss
from recheck.runtime import (
    ResourceBudget,
    RNGStreams,
    atomic_json,
    choose_device,
    model_hash,
    require_rays_gate,
    save_checkpoint,
    tensor_hash,
)


In [ ]:
RUN_TRAINING = False
ALLOW_CPU_EXPLORATION = False        # explicit override for this short student example only
DEVICE = "auto"                     # CUDA -> MPS -> CPU; see doctor and resource budget
CONFIG_PATH = None                  # auto: mnist_mps_registered.yaml if present, else mnist_mvp.yaml
EXPLORATORY_UPDATES = 100
BATCH_SIZE = None                    # None takes the registered student batch/OT support
SEED = 101
LEARNING_RATE = 1e-4
WARMUP = 10
EMA_DECAY = 0.99
WEIGHT_DECAY = 1e-4
GRAD_CLIP = 1.0
TUBE_RHO = 0.10
LAMBDA_W = 1.0
GAMMA = 1.0
DISTRIBUTION_FAMILY = "sinkhorn"      # "sinkhorn" or "swd"
REPRESENTATION = "endpoint"          # "endpoint", "displacement", "joint"
SWD_PROJECTIONS = 128
CHECKPOINT_EVERY = 25
RESUME = False
PREVIEW_SAMPLES = 16
PREVIEW_SEED = 730002                # fixed validation noise, independent of model/NFE
MAKE_GIFS = True
SHOW_WIDGETS = True
OUTPUT_DIR = Path("outputs/mnist_wasserstein_notebook")


In [ ]:
REPOSITORY_URL = "https://github.com/Vovanm88/ddpm_vibe_research.git"
repo_dir = next((p for p in (Path.cwd(), *Path.cwd().parents)
                 if (p / "src" / "recheck").is_dir()), None)
if repo_dir is None:
    repo_dir = Path(tempfile.mkdtemp(prefix="mnist-wasserstein-source-")) / "repository"
    subprocess.check_call(["git", "clone", "--depth", "1", "--branch", GIT_BRANCH,
                           REPOSITORY_URL, str(repo_dir)])
registered = repo_dir / "replication/configs/mnist_mps_registered.yaml"
config_path = Path(CONFIG_PATH).expanduser() if CONFIG_PATH else (
    registered if registered.is_file() else repo_dir / "replication/configs/mnist_mvp.yaml"
)
if not config_path.is_absolute():
    config_path = repo_dir / config_path
config_path = config_path.resolve()
config = load_config(config_path)
config["device"] = DEVICE
for owner, key in ((config, "output_dir"), (config["data"], "root"), (config["resources"], "rays_gate")):
    candidate = Path(owner[key]).expanduser()
    owner[key] = str(candidate if candidate.is_absolute() else (repo_dir / candidate).resolve())
print("Configuration:", config_path)
print("Registered teacher/student updates:", config["teacher"]["updates"], config["student"]["updates"])
print("Notebook training enabled:", RUN_TRAINING)

stages = ["prepare-mnist", "train-evaluator", "train-teacher", "teacher-gate", "calibrate",
          "benchmark", "distill-suite", "evaluate-suite", "report"]
commands = ["cd " + shlex.quote(str(repo_dir)),
    shlex.join([sys.executable, "-m", "recheck.cli", "pilot-rays", "--config", "replication/configs/rays_cpu.yaml"])]
for stage in stages:
    command = shlex.join([sys.executable, "-m", "recheck.cli", stage, "--config", str(config_path)])
    commands.extend([command + " --dry-run", command])
display(Markdown("### Основной зарегистрированный запуск — выполнять по этапам\n\n"
    "До финальной серии оцените warmed benchmark и общий лимит 6 accelerator-hours. "
    "Урезание бюджета регистрируется одинаково для всех методов до сравнения.\n\n```bash\n"
    + "\n".join(commands) + "\n```\n\nДля продолжения training-команд добавьте `--resume`; "
    "остальные команды не начинают скрытое обучение."))


## Сначала обязательный rays gate, затем существующий обученный teacher

Ни dataset factory, ни MNIST download/training не вызываются до `require_rays_gate`.
Если teacher отсутствует, выполняйте показанные CLI-этапы: notebook не подменяет его
случайной сетью. Изменение алгоритма делает старый rays gate недействительным.

In [ ]:
require_rays_gate(config)              # BEFORE any MNIST access or student training
schedule = experiment_schedule(config)
device = choose_device(DEVICE)
torch.set_num_threads(config["threads"])
BATCH_SIZE = config["student"]["batch_size"] if BATCH_SIZE is None else BATCH_SIZE
if BATCH_SIZE < 1 or EXPLORATORY_UPDATES < 1:
    raise ValueError("positive exploratory updates/batch required")
if DISTRIBUTION_FAMILY not in {"sinkhorn", "swd"} or REPRESENTATION not in {"endpoint", "displacement", "joint"}:
    raise ValueError("unknown distribution family or representation")
OUTPUT_DIR = Path(OUTPUT_DIR).expanduser().resolve()

teacher_paths = [Path(config["output_dir"]) / "teacher/ema_model",
    repo_dir / "replication/demo" / config["protocol_id"] / "teacher/ema_model"]
teacher_path = next((p for p in teacher_paths if (p / "config.json").is_file()), None)
teacher, teacher_hash, teacher_gate = None, None, None
if teacher_path is None:
    display(Markdown("**Обученный MNIST teacher пока отсутствует.** Выполните `prepare-mnist`, "
        "`train-evaluator`, `train-teacher` и `teacher-gate` командами выше. "
        "`RUN_TRAINING=False` оставляет training-ячейки выключенными; случайный teacher не создаётся."))
else:
    teacher = load_model(teacher_path, device).eval().requires_grad_(False)
    teacher_hash = model_hash(teacher)
    manifest_path = teacher_path.parent / "manifest.json"
    if manifest_path.is_file():
        teacher_manifest = json.loads(manifest_path.read_text())
        if teacher_manifest.get("status") != "completed" or teacher_manifest.get("teacher_sha256") != teacher_hash:
            raise ValueError("Teacher manifest is incomplete or weights differ from its fixed hash")
    gate_path = teacher_path.parent.parent / "teacher_gate.json"
    if gate_path.is_file():
        teacher_gate = json.loads(gate_path.read_text())
    print("Loaded actual teacher:", teacher_path, "hash:", teacher_hash, "device:", device)
    print("Unconditional U-Net, parameters:", sum(p.numel() for p in teacher.parameters()))


## Открытая локальная математика для полного изображения

При общей ширине `H>0` teacher делает `m` шагов `H/m`; student — один шаг `H`.
В tube-режимах старт изменяется на `rho*H*eta`, и teacher target обязательно
**заново считается из изменённого старта**. Одна OT-точка — всё изображение,
для MNIST `D=784`, а не отдельный pixel. Joint имеет размер `2D`, но тот же `sqrt(D)`.

In [ ]:
@torch.no_grad()
def teacher_microsteps(model, anchor, t, width, microsteps=2):
    state = anchor.detach()
    h = float(width) / microsteps
    for k in range(microsteps):
        state = state - h * model(state, float(t) - k * h).sample
    return state.detach()


def image_transition_features(anchor, endpoint, representation="endpoint", gamma=1.0):
    dimension = endpoint[0].numel()
    a, y = anchor.flatten(1), endpoint.flatten(1)
    if representation == "endpoint":
        return y / math.sqrt(dimension)
    displacement = y - a
    if representation == "displacement":
        return displacement / math.sqrt(dimension)
    if representation == "joint":
        return torch.cat((math.sqrt(gamma) * a, displacement), dim=1) / math.sqrt(dimension)
    raise ValueError("unknown representation")


In [ ]:
# This cell stays inert by default. Existing prepared train data are loaded only
# after rays gate; benchmark_data uses download=False and never opens test here.
train_samples, calibration = None, None
if RUN_TRAINING:
    require_rays_gate(config)
    if teacher is None:
        raise RuntimeError("First train/load the real teacher using the CLI commands above")
    if not teacher_gate or not teacher_gate.get("passed") or teacher_gate.get("teacher_sha256") != teacher_hash:
        raise RuntimeError("Run teacher-gate first: need its passing validation result for these exact weights")
    if device.type == "cpu" and not ALLOW_CPU_EXPLORATION:
        raise RuntimeError("No accelerator selected. Keep training off or explicitly enable the short CPU exploration")
    train_samples = benchmark_data(config, "train")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    # Calibration is frozen teacher-only, 256 clean train anchors per interval;
    # same table for tube/no-tube and all student seeds, no student-dependent scale.
    calibration_budget = ResourceBudget(config, device)
    try:
        calibration = calibrate(config, teacher, train_samples,
                                 output_path=OUTPUT_DIR / "calibration.json")
    finally:
        try:
            calibration_budget.check()
        finally:
            calibration_budget.close()
    print("Train images:", train_samples.shape, "D:", train_samples[0].numel(),
          "OT support:", BATCH_SIZE)


## Видимый student training: одна paired последовательность для всех четырёх режимов

Это короткая exploratory-серия, не основной preregistered результат. Включайте её
только после успешных rays/evaluator/teacher gates. AdamW, `zero_grad`, target,
`loss.backward`, clipping, `step`, EMA и сохранение находятся здесь.

Для Sinkhorn используется сходящаяся squared-cost/KL функция с обеими self-cost;
на MPS float64 OT переносится на CPU через проверенный differentiable bridge.
SWD имеет отдельный RNG, P-normalized squared projections без корня. Дополнительная
случайность не меняет последовательности data/time/noise/tube.

In [ ]:
trained, training_logs, pairing_traces = {}, {}, {}
if RUN_TRAINING:
    require_rays_gate(config)
    if teacher is None or train_samples is None or calibration is None:
        raise RuntimeError("Run the gated teacher/train-only calibration cells first")
    dist_label = "E" if (DISTRIBUTION_FAMILY, REPRESENTATION) == ("sinkhorn", "endpoint") else (
        DISTRIBUTION_FAMILY + "_" + REPRESENTATION + "_"
    )
    method_specs = [("M0", None, 0.0), ("M1", None, TUBE_RHO),
                    (dist_label + "0", DISTRIBUTION_FAMILY, 0.0),
                    (dist_label + "1", DISTRIBUTION_FAMILY, TUBE_RHO)]
    budget = ResourceBudget(config, device)         # same run ledger/one accelerator lock as CLI
    try:
        for method_name, family, rho in method_specs:
            student = copy.deepcopy(teacher).train().requires_grad_(True)
            ema = copy.deepcopy(teacher).eval().requires_grad_(False)
            assert model_hash(student) == teacher_hash == model_hash(ema)
            optimizer = torch.optim.AdamW(student.parameters(), lr=LEARNING_RATE,
                                          betas=(.9, .999), weight_decay=WEIGHT_DECAY)
            streams = RNGStreams(SEED)             # RESET paired streams for every method
            method_dir = OUTPUT_DIR / method_name
            method_dir.mkdir(parents=True, exist_ok=True)
            checkpoint = method_dir / "training.pt"
            contract = {"scope": "notebook_exploratory_one_paired_seed",
                "teacher_sha256": teacher_hash, "train_sha256": tensor_hash(train_samples),
                "calibration_sha256": calibration["sha256"], "schedule": schedule.to_dict(),
                "seed": SEED, "updates": EXPLORATORY_UPDATES, "batch_size": BATCH_SIZE,
                "family": family, "representation": REPRESENTATION, "rho": rho,
                "lambda_w": LAMBDA_W, "gamma": GAMMA, "lr": LEARNING_RATE,
                "warmup": WARMUP, "ema_decay": EMA_DECAY, "weight_decay": WEIGHT_DECAY,
                "grad_clip": GRAD_CLIP, "swd_projections": SWD_PROJECTIONS}
            history, pairing_trace, completed = [], [], 0
            if checkpoint.exists():
                if not RESUME:
                    raise FileExistsError("Existing notebook run: set RESUME=True or choose new OUTPUT_DIR")
                saved = torch.load(checkpoint, map_location="cpu", weights_only=True)
                if saved["contract"] != contract:
                    raise ValueError("Resume contract differs; use a new output directory")
                student.load_state_dict(saved["student"])
                ema.load_state_dict(saved["ema"])
                optimizer.load_state_dict(saved["optimizer"])
                for state in optimizer.state.values():
                    for key, value in state.items():
                        if isinstance(value, torch.Tensor):
                            state[key] = value.to(device)
                streams.load_state_dict(saved["streams"])
                torch.set_rng_state(saved["torch_rng"])
                if device.type == "cuda" and saved["cuda_rng"]:
                    torch.cuda.set_rng_state_all(saved["cuda_rng"])
                completed, history = saved["updates_completed"], saved["history"]
                pairing_trace = saved["pairing_trace"]

            def commit(status):
                save_checkpoint(checkpoint, {"contract": contract,
                    "student": student.state_dict(), "ema": ema.state_dict(),
                    "optimizer": optimizer.state_dict(), "streams": streams.state_dict(),
                    "torch_rng": torch.get_rng_state(),
                    "cuda_rng": torch.cuda.get_rng_state_all() if device.type == "cuda" else [],
                    "history": history, "pairing_trace": pairing_trace,
                    "updates_completed": completed, "status": status})

            commit("running")
            started = time.perf_counter()
            previous_seconds = history[-1]["training_seconds"] if history else 0.
            try:
                for update in range(completed, EXPLORATORY_UPDATES):
                    j = int(torch.randint(schedule.student_steps, (), generator=streams["interval"]))
                    t, H, m = float(schedule.times[j]), float(schedule.widths[j]), schedule.groups[j]
                    indices = torch.randint(len(train_samples), (BATCH_SIZE,), generator=streams["data"])
                    x0 = train_samples[indices].to(device)
                    x1 = torch.randn(x0.shape, generator=streams["noise"]).to(device)
                    eta = torch.randn(x0.shape, generator=streams["tube"]).to(device)  # also rho=0
                    anchor = (1 - t) * x0 + t * x1
                    a_used = anchor + rho * H * eta
                    # Target is RECOMPUTED from the actual perturbed image, not clean anchor.
                    y_teacher = teacher_microsteps(teacher, a_used, t, H, m)
                    y_student = a_used - H * student(a_used, t).sample
                    pair = (y_student - y_teacher.detach()).square().mean()   # mean over B and pixels
                    dist = pair.new_zeros(())
                    if family is not None and LAMBDA_W != 0:
                        phi_student = image_transition_features(a_used, y_student, REPRESENTATION, GAMMA)
                        phi_teacher = image_transition_features(a_used, y_teacher, REPRESENTATION, GAMMA).detach()
                        if family == "sinkhorn":
                            epsilon = calibration["intervals"][j]["representations"][REPRESENTATION]["epsilon"]
                            dist = production_sinkhorn(phi_student.float(), phi_teacher.float(), epsilon)
                        elif family == "swd":
                            dist = sliced_wasserstein_loss(phi_student.float(), phi_teacher.float(),
                                generator=streams["swd"], num_directions=SWD_PROJECTIONS)
                        else:
                            raise ValueError("unknown distribution family")
                    loss = pair + LAMBDA_W * dist
                    if not torch.isfinite(loss):
                        raise FloatingPointError("Non-finite loss invalidates this series")
                    lr = learning_rate(update, EXPLORATORY_UPDATES, WARMUP, LEARNING_RATE)
                    for group in optimizer.param_groups:
                        group["lr"] = lr
                    optimizer.zero_grad(set_to_none=True)
                    loss.backward()
                    grad_norm = torch.nn.utils.clip_grad_norm_(student.parameters(), GRAD_CLIP,
                                                               error_if_nonfinite=True)
                    optimizer.step()
                    with torch.no_grad():
                        for average, parameter in zip(ema.parameters(), student.parameters()):
                            average.lerp_(parameter.detach(), 1 - EMA_DECAY)
                        for average, buffer in zip(ema.buffers(), student.buffers()):
                            average.copy_(buffer)
                    completed = update + 1
                    pairing_trace.append(hashlib.sha256((str(j) + tensor_hash(indices) +
                        tensor_hash(x1) + tensor_hash(eta)).encode()).hexdigest())
                    history.append({"update": completed, "pair_loss": float(pair.detach()),
                        "dist_loss": float(dist.detach()), "total_loss": float(loss.detach()),
                        "grad_norm": float(grad_norm), "lr": lr, "interval": j,
                        "training_seconds": previous_seconds + time.perf_counter() - started})
                    if completed == 1 or completed % CHECKPOINT_EVERY == 0:
                        print(method_name, completed, "pair=", float(pair.detach()), "dist=", float(dist.detach()))
                        commit("running")
                        budget.check()
            except BaseException as error:
                # Retain last fully committed weights/optimizer/RNG; discard partial update.
                saved = torch.load(checkpoint, map_location="cpu", weights_only=True)
                saved["status"] = "interrupted" if isinstance(error, KeyboardInterrupt) else "failed"
                save_checkpoint(checkpoint, saved)
                raise
            commit("completed")
            torch.save(ema.state_dict(), method_dir / "ema_state_dict.pt")
            if hasattr(ema, "save_pretrained"):
                ema.save_pretrained(method_dir / "ema_model", safe_serialization=True)
            atomic_json(method_dir / "manifest.json", contract | {
                "status": "completed", "method": method_name, "ema_sha256": model_hash(ema),
                "updates_completed": completed, "micro_batch": BATCH_SIZE,
                "optimizer_effective_batch": BATCH_SIZE,
                "ot_support_size": BATCH_SIZE if family else 0,
                "training_seconds": history[-1]["training_seconds"], "rng_seeds": streams.seeds})
            atomic_json(method_dir / "history.json", history)
            trained[method_name], training_logs[method_name] = ema, history
            pairing_traces[method_name] = pairing_trace
            assert model_hash(teacher) == teacher_hash and all(p.grad is None for p in teacher.parameters())
        assert all(trace == pairing_traces["M0"] for trace in pairing_traces.values())
        print("All paired data/time/noise/tube traces match; parent weights unchanged.")
    finally:
        try:
            budget.check()
        finally:
            budget.close()
else:
    print("Training remains off. Inspect the visible loop, then explicitly enable the short exploration when ready.")


## Фиксированные изображения и реальные denoising states

Preview использует один исходный шум для teacher N32, teacher N16 и всех появившихся
учеников. Seed не зависит от числа шагов. Изображения ниже одинаково преобразуются
`(raw_FM+1)/2` и clamp; до clamp показывается доля значений вне диапазона.
Это preview/fidelity, а не основная feature-MMD quality: независимый CNN и полная
оценка относительно real test остаются в CLI `evaluate-suite`.

In [ ]:
def image_mosaic(raw, columns=4):
    pixels, diagnostics = evaluator_preprocess(raw)
    array = pixels.detach().cpu().numpy()
    rows = math.ceil(len(array) / columns)
    height, width = array.shape[-2:]
    canvas = np.ones((rows * (height + 2) + 2, columns * (width + 2) + 2), dtype=np.float32)
    for i, image in enumerate(array):
        row, col = divmod(i, columns)
        top, left = 2 + row * (height + 2), 2 + col * (width + 2)
        canvas[top:top + height, left:left + width] = image[0]
    return canvas, diagnostics


previews = {}
if teacher is not None:
    require_rays_gate(config)
    preview_noise = torch.randn((PREVIEW_SAMPLES, *teacher.sample_shape),
                                generator=torch.Generator().manual_seed(PREVIEW_SEED))
    preview_models = {f"Original N{schedule.teacher_steps}": (teacher, schedule.teacher_steps, None),
                      f"Original N{schedule.student_steps}": (teacher, schedule.student_steps, schedule.times)}
    preview_models.update({f"{name} N{schedule.student_steps}": (model, schedule.student_steps, schedule.times)
                           for name, model in trained.items()})
    preview_budget = ResourceBudget(config, device)
    try:
        for name, (model, steps, times) in preview_models.items():
            previews[name] = sample_euler(model, preview_noise, times=times, steps=steps, record_history=True,
                                          batch_size=PREVIEW_SAMPLES)
            preview_budget.check()
    finally:
        try:
            preview_budget.check()
        finally:
            preview_budget.close()
    assert len({result["initial_noise_sha256"] for result in previews.values()}) == 1
    fig, axes = plt.subplots(1, len(previews), figsize=(4 * len(previews), 4), squeeze=False,
                              constrained_layout=True)
    for ax, (name, result) in zip(axes.flat, previews.items()):
        mosaic, diagnostics = image_mosaic(result["samples"])
        ax.imshow(mosaic, cmap="gray", vmin=0, vmax=1, interpolation="nearest")
        ax.set_title(name + f"\noutside [0,1]: {diagnostics['out_of_range_fraction']:.2%}")
        ax.axis("off")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(OUTPUT_DIR / "fixed_samples.png", dpi=150)
    fig.savefig(OUTPUT_DIR / "fixed_samples.svg")
    plt.show()
    original = next(iter(previews.values()))["samples"]
    fidelity = {name: {"paired_endpoint_mse": float((result["samples"] - original).square().mean()),
        "nfe": result["nfe"], "initial_noise_sha256": result["initial_noise_sha256"],
        **evaluator_preprocess(result["samples"])[1]} for name, result in previews.items()}
    atomic_json(OUTPUT_DIR / "preview_fidelity.json", fidelity)
    torch.save({"noise": preview_noise, "previews": previews}, OUTPUT_DIR / "preview_states.pt")
    print("Preview/fidelity:", fidelity)
else:
    print("Preview waits for an actual trained teacher; no random-model images are substituted.")


In [ ]:
if previews and MAKE_GIFS:
    gif_paths = []
    for name, result in previews.items():
        history, times = result["history"], result["times"]
        figure, ax = plt.subplots(figsize=(5, 5))
        mosaic, _ = image_mosaic(history[0])
        artist = ax.imshow(mosaic, cmap="gray", vmin=0, vmax=1, interpolation="nearest")
        ax.axis("off")

        def frame(index):
            mosaic, diagnostic = image_mosaic(history[index])
            artist.set_data(mosaic)
            ax.set_title(f"{name} · actual step {index}/{len(history)-1} · t={float(times[index]):.4f}\n"
                         f"raw out-of-range pixels: {diagnostic['out_of_range_fraction']:.2%}")
            return (artist,)

        movie = animation.FuncAnimation(figure, frame, frames=range(len(history)), interval=120)
        path = OUTPUT_DIR / (name.replace(" ", "_") + "_denoising.gif")
        movie.save(path, writer=animation.PillowWriter(fps=8), dpi=80)
        plt.close(figure)
        gif_paths.append(str(path))
    display(Image(filename=gif_paths[-1]))
    print("Actual-state image-grid GIFs:", gif_paths)

if previews and SHOW_WIDGETS:
    selector = widgets.Dropdown(options=list(previews), description="Model")
    slider = widgets.IntSlider(value=0, min=0, max=schedule.teacher_steps, step=1,
                               description="Euler step", continuous_update=False)
    play = widgets.Play(value=0, min=0, max=schedule.teacher_steps, step=1, interval=120)
    link = widgets.jslink((play, "value"), (slider, "value"))
    canvas = widgets.Output()

    def render_step(change=None):
        result = previews[selector.value]
        slider.max = play.max = len(result["history"]) - 1
        index = min(slider.value, slider.max)
        mosaic, diagnostic = image_mosaic(result["history"][index])
        with canvas:
            clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(5, 5))
            ax.imshow(mosaic, cmap="gray", vmin=0, vmax=1, interpolation="nearest")
            ax.set_title(f"{selector.value} · step {index}/{slider.max} · t={float(result['times'][index]):.4f}\n"
                         f"raw out-of-range: {diagnostic['out_of_range_fraction']:.2%}")
            ax.axis("off")
            plt.show()

    selector.observe(render_step, names="value")
    slider.observe(render_step, names="value")
    display(widgets.VBox([widgets.HBox([selector, play, slider]), canvas]))
    render_step()


Для смены loss используйте `DISTRIBUTION_FAMILY="swd"`, `REPRESENTATION="joint"` или
`"displacement"` и новый `OUTPUT_DIR`. `LAMBDA_W=0` даёт paired MSE-control.
Для нового числа шагов изменяйте зарегистрированную отдельную сетку, заново калибруйте
teacher и стартуйте student из исходного parent: не выбирайте сетку по удачным цифрам.

Model architecture находится в `config["model"]`; замена только student на другую
архитектуру является отдельной compression-абляцией. Основной протокол сохраняет общую
U-Net архитектуру teacher/student. Optimizer и loop здесь обычные PyTorch; native веса
сохраняются через Diffusers ModelMixin, checkpoint можно безопасно загрузить с
`torch.load(..., weights_only=True)`. Для продолжения того же exploration поставьте
`RESUME=True`, не меняя его контракт.

Подтверждающая таблица — original N32, original N16 и M0/M1/E0/E1 N16 на трёх seed.
Качество к данным, fidelity к учителю и стоимость показываются отдельно. Два красивых
sample grids и небольшой training loss не доказывают добавочную пользу OT.